# Kendra-voice LoRA — Gemma 3n-E2B (Unsloth, free Colab T4)

Trains Kendra's personality into her brain's weights per
`docs/UNSLOTH_FINETUNING_PLAN.md` (candidate C1).

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then upload
`kendra_voice_sft.jsonl` (from `exports/finetune/` on the iMac — the expanded
one produced by `scripts/synthesize_finetune_dialogs.py`) when the upload
cell prompts. Run all cells top to bottom; total time ≈ 30-60 min.

**Output:** `kendra-qwen3-1.7b-v1.Q4_K_M.gguf` — downloaded at the end,
drops into `models/gemma3n-kendra-v1/` on the iMac.

In [ ]:
# MUST run before anything imports unsloth/torch.
# torch.compile (Dynamo) crashes on unsloth's Gemma-3n patches
# (Gemma3nRMSNorm has no 'weight'); eager mode trains correctly.
import os
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"] = "1"
import torch._dynamo
torch._dynamo.config.suppress_errors = True
print("eager mode forced — safe for Gemma 3n")


In [ ]:
%%capture
# Unsloth + deps (Colab T4). Pinned to the install unsloth's Colab notebooks use.
!pip install unsloth
!pip install --no-deps --upgrade timm  # gemma-3n vision tower dep, harmless for text-only

In [ ]:
from unsloth import FastLanguageModel
import torch

# Qwen3-1.7B: plan candidate C4. Gemma 3n's unsloth compatibility patches
# are broken in current Colab environments (Gemma3nRMSNorm AttributeError
# in unsloth_zoo, compiled AND eager) — Qwen3 support is mature and clean.
# Qwen3-1.7B is already Kendra's on-disk fallback brain; deploy is one
# KENDRA_LLM_MODEL swap.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-1.7B",
    max_seq_length=1024,
    load_in_4bit=True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
# Upload the dataset exported from her brain.
from google.colab import files
uploaded = files.upload()  # choose kendra_voice_sft.jsonl
DATASET_PATH = next(iter(uploaded))
print("using", DATASET_PATH)

In [ ]:
import json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen3")

rows = [json.loads(line) for line in open(DATASET_PATH, encoding="utf-8")]
print(f"{len(rows)} examples")

def to_text(example):
    return {"text": tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False)}

dataset = Dataset.from_list(rows).map(to_text)
dataset = dataset.train_test_split(test_size=0.1, seed=3407)
dataset

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,   # effective batch 8
        num_train_epochs=2,              # plan: <=2, watch eval loss
        learning_rate=2e-4,
        warmup_ratio=0.05,
        weight_decay=0.01,
        lr_scheduler_type="linear",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        optim="adamw_8bit",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)
# Loss only on Kendra's replies, not Jonathan's words.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
stats = trainer.train()
# Healthy landing zone per the plan: final train loss ~0.5-1.0.
stats

In [ ]:
# Register smoke test before exporting — she should sound like Kendra.
from transformers import TextStreamer
FastLanguageModel.for_inference(model)
for probe in [
    "Hey, can you hear me?",
    "How are you feeling this morning?",
    "What do you think about heavy metal?",
]:
    messages = [{"role": "user", "content": probe}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt").to("cuda")
    print(f"\n=== {probe}")
    _ = model.generate(input_ids=inputs, max_new_tokens=60,
                       temperature=0.7, top_p=0.8, top_k=20,
                       streamer=TextStreamer(tokenizer, skip_prompt=True))

In [ ]:
# Export merged GGUF Q4_K_M for llama.cpp (iMac + Pi, same file).
model.save_pretrained_gguf(
    "kendra-qwen3-1.7b-v1",
    tokenizer,
    quantization_method="q4_k_m",
)
!ls -la kendra-qwen3-1.7b-v1

In [ ]:
# Also save the tiny LoRA adapter (~100MB) as the re-trainable artifact.
model.save_pretrained("kendra-voice-lora-v1")
tokenizer.save_pretrained("kendra-voice-lora-v1")
!zip -qr kendra-voice-lora-v1.zip kendra-voice-lora-v1

In [ ]:
from google.colab import files
import glob

# Newer unsloth writes the GGUF to a sibling "<name>_gguf/" directory;
# older versions put it inside the model directory. Look in both.
candidates = glob.glob("kendra-qwen3-1.7b-v1_gguf/*.gguf") + glob.glob(
    "kendra-qwen3-1.7b-v1/*.gguf"
)
assert candidates, "no GGUF produced - check the export cell output"
files.download(candidates[0])                # the deployable brain
files.download("kendra-voice-lora-v1.zip")   # the adapter, for future rounds


## Back on the iMac

```
mkdir -p models/gemma3n-kendra-v1
mv ~/Downloads/*.Q4_K_M.gguf models/gemma3n-kendra-v1/
KENDRA_LLM_MODEL=models/gemma3n-kendra-v1/<file>.gguf scripts/start_llm_intel_macos.sh
```

Then swap in the shrunk charter (`charter/charter-finetuned.md` →
`charter/charter.md`, keep a backup) and run the register + timing probes.
Rollback = restore the charter and drop the env var.